# Optimization Algorithms: Genetic vs. Gradient Descent

In this notebook, we'll implement two optimization algorithms from scratch using NumPy:
1. **Gradient Descent** - A calculus-based optimizer
2. **Genetic Algorithm** - A biology-inspired optimizer

We'll apply both to solve a **linear regression** problem and compare their performance.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

## 1. Load and Explore the Data

We'll predict **Energy Consumption** based on building features.

In [ ]:
# Load data
df = pd.read_csv('test_energy_data.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print("\nFirst few rows:")
df.head(5)

In [ ]:
# Basic statistics
df.describe()

## 2. Prepare the Data

For simplicity, we'll use only **numerical features** to predict Energy Consumption.

In [ ]:
# Select numerical features
features = ['Square Footage', 'Number of Occupants', 'Appliances Used', 'Average Temperature']
target = 'Energy Consumption'

# Extract features and target
X = df[features].values
y = df[target].values

print(f"Feature matrix X shape: {X.shape}")
print(f"Target vector y shape: {y.shape}")
print(f"\nSample input (first building):")
print(X[0])
print(f"\nTarget output: {y[0]:.2f} kWh")

### Normalize Features

**Why normalize?** Features have different scales (e.g., square footage vs. occupants). Normalization helps our optimizers finish ('converge') more quickly. QUESTION: why?

Also, normalisation means we don't have issues if we convert the units of a feature (ex: sqft to m^2).

In [ ]:
# Normalize features (mean=0, std=1)
# QUESTION: why did we compute the means/stds on axis 0?
X_mean = np.mean(X, axis=0)
X_std = np.std(X, axis=0)
X_normalized = (X - X_mean) / X_std

# Normalize target (mean=0, std=1)
# CHANGED: Added target normalization to help Genetic Algorithm converge
y_mean = np.mean(y)
y_std = np.std(y)
y_normalized = (y - y_mean) / y_std

print("Before normalization:")
print(f"Mean: {np.mean(X, axis=0)}")
print(f"Std: {np.std(X, axis=0)}")

print("\nAfter normalization:")
print(f"Mean: {np.mean(X_normalized, axis=0)}")
print(f"Std: {np.std(X_normalized, axis=0)}")

### Train-Test Split

We split data into:
- **Training set** (80%): Used to learn parameters
- **Test set** (20%): Used to evaluate performance on unseen data

QUESTION: why can't we just use all the data for training?

In [ ]:
# Split data (80% train, 20% test)
split_idx = int(0.8 * len(X_normalized))

X_train = X_normalized[:split_idx]
# CHANGED: Use normalized target for training
y_train = y_normalized[:split_idx]

X_test = X_normalized[split_idx:]
# CHANGED: Use normalized target for testing
y_test = y_normalized[split_idx:]

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

## 3. Linear Regression Model

### Prediction Equation

For linear regression, we predict:

$$\hat{y} = w_1 x_1 + w_2 x_2 + ... + w_n x_n + b$$

Or in matrix form:

$$\hat{y} = X \cdot w + b$$

Where:
- $X$ = feature matrix (n_samples × n_features)
- $w$ = weights (n_features,)
- $b$ = bias (scalar)
- $\hat{y}$ = predictions (n_samples,)

### Loss Function: Mean Squared Error (MSE)

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

MSE measures the average squared difference between predictions and actual values.

In [ ]:
def predict(X, weights, bias):
    """
    Make predictions using linear regression.
    
    Parameters:
    - X: Feature matrix (n_samples, n_features)
    - weights: Weight vector (n_features,)
    - bias: Bias scalar
    
    Returns:
    - predictions: (n_samples,)
    """
    return X @ weights + bias


def mse_loss(y_true, y_pred):
    """
    Calculate Mean Squared Error.
    
    Parameters:
    - y_true: Actual values (n_samples,)
    - y_pred: Predicted values (n_samples,)
    
    Returns:
    - MSE loss (scalar)
    """
    return np.mean((y_true - y_pred) ** 2)


# Test with random weights
n_features = X_train.shape[1]
test_weights = np.random.randn(n_features)
test_bias = 0.0

test_predictions = predict(X_train[:5], test_weights, test_bias)
print("Test predictions (first 5):", test_predictions)
print("Actual values (first 5):", y_train[:5])
print(f"\nMSE loss: {mse_loss(y_train[:5], test_predictions):.2f}")
print("Read: random weights give poor predictions.")

## 4. Gradient Descent Optimizer

Gradient descent updates parameters by moving in the direction that reduces loss:

$$w = w - \alpha \cdot \frac{\partial \text{MSE}}{\partial w}$$

$$b = b - \alpha \cdot \frac{\partial \text{MSE}}{\partial b}$$

Where $\alpha$ is the **learning rate**.

### Gradients for MSE Loss:

We can apply chain rule to derive gradients:
$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

$$\hat{y} = X \cdot w + b$$

Results:

$$\frac{\partial \text{MSE}}{\partial w} = \frac{-2}{n} X^T (y - \hat{y})$$

$$\frac{\partial \text{MSE}}{\partial b} = \frac{-2}{n} \sum (y - \hat{y})$$

In [ ]:
def gradient_descent(X, y, learning_rate=0.01, iterations=1000):
    """
    Optimize weights and bias using gradient descent.
    
    Parameters:
    - X: Feature matrix (n_samples, n_features)
    - y: Target values (n_samples,)
    - learning_rate: Step size for updates
    - iterations: Number of optimization steps
    
    Returns:
    - weights, bias, loss_history
    """
    n_samples, n_features = X.shape
    
    # Initialize parameters
    weights = np.random.randn(n_features) * 0.01
    bias = 0.0
    
    loss_history = []
    
    for i in range(iterations):
        # Forward pass: make predictions
        y_pred = predict(X, weights, bias)
        
        # Calculate loss
        loss = mse_loss(y, y_pred)
        loss_history.append(loss)
        
        # Compute gradients
        error = y_pred - y
        dw = (2 / n_samples) * (X.T @ error)
        db = (2 / n_samples) * np.sum(error)
        
        # Update parameters
        weights -= learning_rate * dw
        bias -= learning_rate * db
        
        # Print progress
        if (i + 1) % 200 == 0:
            print(f"Iteration {i+1}/{iterations}, Loss: {loss:.2f}")
    
    return weights, bias, loss_history

In [ ]:
# Train with gradient descent
print("Training with Gradient Descent...")
gd_weights, gd_bias, gd_loss_history = gradient_descent(
    X_train, y_train, 
    learning_rate=0.1, 
    iterations=1000
)

print(f"\nFinal weights: {gd_weights}")
print(f"Final bias: {gd_bias:.2f}")

## 5. Genetic Algorithm Optimizer

Genetic algorithms are inspired by natural selection:

1. **Population**: Create random candidate solutions
2. **Fitness**: Evaluate each candidate (lower MSE = higher fitness)
3. **Selection**: Keep the best candidates
4. **Crossover**: Combine pairs of good solutions
5. **Mutation**: Randomly modify some solutions
6. **Repeat**: Evolve over generations

### Key Differences from Gradient Descent:
- No gradients needed (works for any loss function)
- Explores solution space more broadly
- Less likely to get stuck in local minima
- Usually slower to converge

In [ ]:
def fitness(individual, X, y):
    """
    Calculate fitness (negative MSE, since we want to maximize fitness).
    
    Parameters:
    - individual: [weights..., bias] concatenated
    - X: Feature matrix
    - y: Target values
    
    Returns:
    - Negative MSE (higher is better)
    """
    weights = individual[:-1]
    bias = individual[-1]
    y_pred = predict(X, weights, bias)
    return -mse_loss(y, y_pred)  # Negative because we maximize fitness


def genetic_algorithm(X, y, population_size=100, generations=100, mutation_rate=0.1):
    """
    Optimize weights and bias using genetic algorithm.
    
    Parameters:
    - X: Feature matrix
    - y: Target values
    - population_size: Number of candidate solutions
    - generations: Number of evolution iterations
    - mutation_rate: Probability of random changes
    
    Returns:
    - weights, bias, loss_history
    """
    n_features = X.shape[1]
    n_params = n_features + 1  # weights + bias
    
    # Initialize population (random solutions)
    population = np.random.randn(population_size, n_params) * 0.1
    
    loss_history = []
    
    for gen in range(generations):
        # Evaluate fitness for all individuals
        fitnesses = np.array([fitness(ind, X, y) for ind in population])
        
        # Track best loss (convert fitness back to MSE)
        best_fitness = np.max(fitnesses)
        loss_history.append(-best_fitness)  # Convert back to positive MSE
        
        # Selection: keep top 50%
        top_indices = np.argsort(fitnesses)[-population_size//2:]
        parents = population[top_indices]
        
        # Crossover: create offspring
        offspring = []
        for _ in range(population_size // 2):
            # Select two random parents
            parent1, parent2 = parents[np.random.choice(len(parents), 2, replace=False)]
            
            # Crossover: average of parents
            child = (parent1 + parent2) / 2
            offspring.append(child)
        
        offspring = np.array(offspring)
        
        # Mutation: add random noise
        mutation_mask = np.random.random(offspring.shape) < mutation_rate
        offspring[mutation_mask] += np.random.randn(np.sum(mutation_mask)) * 0.1
        
        # New population: parents + offspring
        population = np.vstack([parents, offspring])
        
        # Print progress
        if (gen + 1) % 20 == 0:
            print(f"Generation {gen+1}/{generations}, Best Loss: {-best_fitness:.2f}")
    
    # Return best individual
    fitnesses = np.array([fitness(ind, X, y) for ind in population])
    best_individual = population[np.argmax(fitnesses)]
    
    weights = best_individual[:-1]
    bias = best_individual[-1]
    
    return weights, bias, loss_history

In [ ]:
# Train with genetic algorithm
print("Training with Genetic Algorithm...")
ga_weights, ga_bias, ga_loss_history = genetic_algorithm(
    X_train, y_train,
    population_size=100,
    generations=100,
    mutation_rate=0.1
)

print(f"\nFinal weights: {ga_weights}")
print(f"Final bias: {ga_bias:.2f}")

## 6. Compare Results

Let's visualize the **learning curves** (loss over time) for both algorithms.

In [ ]:
# Plot learning curves
plt.figure(figsize=(12, 5))

# Gradient Descent
plt.subplot(1, 2, 1)
plt.plot(gd_loss_history, label='Gradient Descent', color='blue', linewidth=2)
plt.xlabel('Iteration', fontsize=12)
plt.ylabel('MSE Loss', fontsize=12)
plt.title('Gradient Descent Learning Curve', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.legend()

# Genetic Algorithm
plt.subplot(1, 2, 2)
plt.plot(ga_loss_history, label='Genetic Algorithm', color='green', linewidth=2)
plt.xlabel('Generation', fontsize=12)
plt.ylabel('MSE Loss', fontsize=12)
plt.title('Genetic Algorithm Learning Curve', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

print(f"Gradient Descent - Final Train Loss: {gd_loss_history[-1]:.2f}")
print(f"Genetic Algorithm - Final Train Loss: {ga_loss_history[-1]:.2f}")

### Evaluate on Test Set

The **test set** shows how well our models generalize to unseen data.

In [ ]:
# Predictions on test set
gd_test_pred = predict(X_test, gd_weights, gd_bias)
ga_test_pred = predict(X_test, ga_weights, ga_bias)

# Calculate test loss
gd_test_loss = mse_loss(y_test, gd_test_pred)
ga_test_loss = mse_loss(y_test, ga_test_pred)

print("Test Set Performance:")
print("=" * 50)
print(f"Gradient Descent - Test MSE: {gd_test_loss:.2f}")
print(f"Genetic Algorithm - Test MSE: {ga_test_loss:.2f}")
print("\nInterpretation:")
# Denormalize RMSE to get back to kWh
# CHANGED: Denormalize the error to interpret in original units (kWh)
gd_rmse = np.sqrt(gd_test_loss) * y_std
ga_rmse = np.sqrt(ga_test_loss) * y_std
print(f"On average, GD predictions are off by ±{gd_rmse:.2f} kWh")
print(f"On average, GA predictions are off by ±{ga_rmse:.2f} kWh")

### Visualize Predictions vs. Actual

In [ ]:
# Denormalize for plotting
# CHANGED: Denormalize predictions and actual values for plotting
y_test_orig = y_test * y_std + y_mean
gd_pred_orig = gd_test_pred * y_std + y_mean
ga_pred_orig = ga_test_pred * y_std + y_mean

# Plot predictions vs actual
plt.figure(figsize=(14, 5))

# Gradient Descent
plt.subplot(1, 2, 1)
plt.scatter(y_test_orig, gd_pred_orig, alpha=0.6, color='blue', edgecolors='black')
plt.plot([y_test_orig.min(), y_test_orig.max()], [y_test_orig.min(), y_test_orig.max()], 
         'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel('Actual Energy Consumption (kWh)', fontsize=12)
plt.ylabel('Predicted Energy Consumption (kWh)', fontsize=12)
plt.title('Gradient Descent: Predictions vs Actual', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)

# Genetic Algorithm
plt.subplot(1, 2, 2)
plt.scatter(y_test_orig, ga_pred_orig, alpha=0.6, color='green', edgecolors='black')
plt.plot([y_test_orig.min(), y_test_orig.max()], [y_test_orig.min(), y_test_orig.max()], 
         'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel('Actual Energy Consumption (kWh)', fontsize=12)
plt.ylabel('Predicted Energy Consumption (kWh)', fontsize=12)
plt.title('Genetic Algorithm: Predictions vs Actual', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Summary and Key Takeaways

### What We Learned:

1. **Linear Regression Components:**
   - **Prediction equation**: $\hat{y} = X \cdot w + b$
   - **Loss function**: MSE measures prediction error
   - **Optimization**: Finding best $w$ and $b$ to minimize loss

2. **Gradient Descent:**
   - Uses calculus (gradients) to find optimal parameters
   - Fast and efficient for smooth loss functions
   - Requires differentiable loss function
   - Can get stuck in local minima

3. **Genetic Algorithm:**
   - Inspired by evolution and natural selection
   - Works with any loss function (no gradients needed)
   - Better at exploring solution space
   - Usually slower to converge

4. **Train vs. Test:**
   - **Training set**: Used to learn parameters
   - **Test set**: Evaluates generalization to unseen data
   - Good test performance means the model generalizes well

### Performance Comparison:

| Algorithm | Speed | Final Loss | Best For |
|-----------|-------|------------|----------|
| Gradient Descent | Fast | Usually lower | Smooth, differentiable problems |
| Genetic Algorithm | Slower | Comparable | Complex, non-differentiable problems |

### When to Use Each:

- **Gradient Descent**: Standard choice for ML (fast, efficient, used in neural networks)
- **Genetic Algorithm**: Useful when:
  - Loss function is not differentiable
  - Many local minima exist
  - You need to explore solution space broadly
  - Used in robotics, game AI, and engineering optimization

### Next Steps:

- Try different learning rates for gradient descent
- Experiment with population size and mutation rate for genetic algorithm
- Add more features to improve predictions
- Try non-linear models (polynomial regression, neural networks)
- Learn about other optimizers (Adam, RMSprop, SGD variants)

## Bonus: Visualize Weight Importance

In [ ]:
# Compare learned weights
feature_names = features

x_pos = np.arange(len(feature_names))
width = 0.35

plt.figure(figsize=(12, 5))
plt.bar(x_pos - width/2, gd_weights, width, label='Gradient Descent', color='blue', alpha=0.7)
plt.bar(x_pos + width/2, ga_weights, width, label='Genetic Algorithm', color='green', alpha=0.7)

plt.xlabel('Features', fontsize=12)
plt.ylabel('Weight Value', fontsize=12)
plt.title('Feature Weights Comparison', fontsize=14, fontweight='bold')
plt.xticks(x_pos, feature_names, rotation=45, ha='right')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("\nWeight Interpretation:")
print("Positive weight = feature increases energy consumption")
print("Negative weight = feature decreases energy consumption")
print("Larger magnitude = stronger effect")